In [0]:
# ============================================================
# CELLULE 0 — CHEMINS
# Toujours exécuter en premier absolu
# ============================================================
PATH_AI4I     = "/Volumes/workspace/default/bronze/ai4i2020.csv"
PATH_MAINT    = "/Volumes/workspace/default/bronze/predictive_maintenance_v3.csv"
PATH_SILVER   = "/Volumes/workspace/default/bronze/silver/sensor_clean/"
PATH_FEATURES = "/Volumes/workspace/default/bronze/silver/sensor_features/"

print('✅ Chemins OK')

✅ Chemins OK


In [0]:
# ============================================================
# CELLULE 1 — IMPORTS
# DOIT être avant toute lecture de données
# Sans ça, F.col(), F.when() etc. ne sont pas reconnus
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

print('✅ Imports PySpark OK')

✅ Imports PySpark OK


In [0]:
# ============================================================
# CELLULE 2 — VÉRIFICATION FICHIER BRONZE
# On vérifie que le CSV est bien présent avant de le lire
# ============================================================
print('Fichiers disponibles dans Bronze :')
for f in dbutils.fs.ls('/Volumes/workspace/default/bronze/'):
    print(f'  {f.name:50s} {f.size} bytes')

Fichiers disponibles dans Bronze :
  ai4i2020.csv                                       522048 bytes
  predictive_maintenance_v3.csv                      2169916 bytes
  silver/                                            0 bytes


In [0]:
# ============================================================
# CELLULE 3 — LECTURE BRONZE
# inferSchema = Spark devine automatiquement les types (int, float...)
# header = True = première ligne = noms de colonnes
# ============================================================
df_raw = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(PATH_AI4I)

print(f'Lignes brutes : {df_raw.count()}')
print(f'Colonnes      : {len(df_raw.columns)}')
df_raw.printSchema()
df_raw.show(5)

Lignes brutes : 10000
Colonnes      : 14
root
 |-- UDI: integer (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Air temperature [K]: double (nullable = true)
 |-- Process temperature [K]: double (nullable = true)
 |-- Rotational speed [rpm]: integer (nullable = true)
 |-- Torque [Nm]: double (nullable = true)
 |-- Tool wear [min]: integer (nullable = true)
 |-- Machine failure: integer (nullable = true)
 |-- TWF: integer (nullable = true)
 |-- HDF: integer (nullable = true)
 |-- PWF: integer (nullable = true)
 |-- OSF: integer (nullable = true)
 |-- RNF: integer (nullable = true)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+------------

In [0]:
# ============================================================
# CELLULE 4 — RENOMMAGE COLONNES
# Règle pro : snake_case, pas d'espaces, pas de symboles [K] [rpm]
# Sinon les requêtes SQL et DAX deviennent un cauchemar
# ============================================================
df = df_raw \
    .withColumnRenamed('UDI',                      'sensor_id') \
    .withColumnRenamed('Product ID',               'product_id') \
    .withColumnRenamed('Type',                     'station_type') \
    .withColumnRenamed('Air temperature [K]',      'air_temp_k') \
    .withColumnRenamed('Process temperature [K]',  'process_temp_k') \
    .withColumnRenamed('Rotational speed [rpm]',   'rotation_speed_rpm') \
    .withColumnRenamed('Torque [Nm]',              'torque_nm') \
    .withColumnRenamed('Tool wear [min]',          'tool_wear_min') \
    .withColumnRenamed('Machine failure',          'machine_failure') \
    .withColumnRenamed('TWF',                      'tool_wear_failure') \
    .withColumnRenamed('HDF',                      'heat_dissipation_failure') \
    .withColumnRenamed('PWF',                      'power_failure') \
    .withColumnRenamed('OSF',                      'overstrain_failure') \
    .withColumnRenamed('RNF',                      'random_failure')

print('✅ Colonnes renommées :')
for c in df.columns:
    print(f'  → {c}')

✅ Colonnes renommées :
  → sensor_id
  → product_id
  → station_type
  → air_temp_k
  → process_temp_k
  → rotation_speed_rpm
  → torque_nm
  → tool_wear_min
  → machine_failure
  → tool_wear_failure
  → heat_dissipation_failure
  → power_failure
  → overstrain_failure
  → random_failure


In [0]:
# ============================================================
# CELLULE 5 — RAPPORT QUALITÉ BRONZE (avant nettoyage)
# On documente l'état initial : nulls + stats descriptives
# En entreprise c'est obligatoire pour le Data Quality SLA
# ============================================================
total = df.count()

print(f"{'='*40}")
print('RAPPORT QUALITÉ — BRONZE')
print(f"{'='*40}")
print(f'Total lignes : {total}\n')

print('Valeurs nulles par colonne :')
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

numeric_cols = ['air_temp_k','process_temp_k','rotation_speed_rpm','torque_nm','tool_wear_min']
print('Statistiques descriptives :')
df.select(numeric_cols).describe().show()

RAPPORT QUALITÉ — BRONZE
Total lignes : 10000

Valeurs nulles par colonne :
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+------------------------+-------------+------------------+--------------+
|sensor_id|product_id|station_type|air_temp_k|process_temp_k|rotation_speed_rpm|torque_nm|tool_wear_min|machine_failure|tool_wear_failure|heat_dissipation_failure|power_failure|overstrain_failure|random_failure|
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+------------------------+-------------+------------------+--------------+
|        0|         0|           0|         0|             0|                 0|        0|            0|              0|                0|                       0|            0|                 0|             0|
+---------+----------+------------+----------+--------------+---------------

In [0]:
# ============================================================
# CELLULE 6 — SUPPRESSION VALEURS PHYSIQUEMENT IMPOSSIBLES
# Température < 250K = -23°C impossible pour une machine industrielle
# Vitesse négative = erreur capteur
# Ces lignes sont corrompues → on les retire proprement
# ============================================================
df_clean = df.filter(
    (F.col('air_temp_k').between(250, 400))            &
    (F.col('process_temp_k').between(250, 400))        &
    (F.col('rotation_speed_rpm').between(100, 5000))   &
    (F.col('torque_nm').between(0, 200))               &
    (F.col('tool_wear_min') >= 0)
)

lignes_supprimees = total - df_clean.count()
retention = round(df_clean.count() / total * 100, 2)

print(f'Lignes supprimées (valeurs impossibles) : {lignes_supprimees}')
print(f'Taux de rétention                       : {retention}%')

Lignes supprimées (valeurs impossibles) : 0
Taux de rétention                       : 100.0%


In [0]:
# ============================================================
# CELLULE 7 — TRAITEMENT OUTLIERS (méthode IQR)
# IQR = Inter-Quartile Range
# On plafonne (clip) les valeurs extrêmes au lieu de les supprimer
# → on garde la ligne mais on corrige la valeur aberrante
# ============================================================
def clip_outliers_iqr(df, col_name):
    q1, q3 = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
    iqr     = q3 - q1
    lower   = q1 - 1.5 * iqr
    upper   = q3 + 1.5 * iqr

    print(f'{col_name:30s} | borne_basse={round(lower,2):8} | borne_haute={round(upper,2):8}')

    return df.withColumn(
        col_name,
        F.when(F.col(col_name) < lower, lower)
         .when(F.col(col_name) > upper, upper)
         .otherwise(F.col(col_name))
    )

print(f"{'Colonne':30s} | {'Borne basse':>12} | {'Borne haute':>12}")
print('-' * 60)

for col in ['air_temp_k', 'process_temp_k', 'rotation_speed_rpm', 'torque_nm']:
    df_clean = clip_outliers_iqr(df_clean, col)

print('\n✅ Outliers traités par IQR')

Colonne                        |  Borne basse |  Borne haute
------------------------------------------------------------
air_temp_k                     | borne_basse=  293.65 | borne_haute=  306.05
process_temp_k                 | borne_basse=   305.5 | borne_haute=   314.3
rotation_speed_rpm             | borne_basse=  1147.0 | borne_haute=  1883.0
torque_nm                      | borne_basse=   13.25 | borne_haute=   66.45

✅ Outliers traités par IQR


In [0]:
# ============================================================
# CELLULE 8 — TIMESTAMP SYNTHÉTIQUE
# Le dataset n'a pas de date réelle → on en crée une artificielle
# sensor_id = numéro de mesure → on l'utilise comme offset en secondes
# Résultat : une timeline continue depuis le 01/01/2020
# ============================================================
df_clean = df_clean.withColumn(
    'timestamp',
    F.expr(
        "timestamp('2020-01-01 00:00:00') + "
        "make_interval(0, 0, 0, 0, 0, sensor_id)"
    )
).orderBy('timestamp')

print('✅ Timestamps générés :')
df_clean.select('sensor_id', 'timestamp').show(10)

✅ Timestamps générés :
+---------+-------------------+
|sensor_id|          timestamp|
+---------+-------------------+
|        1|2020-01-01 00:01:00|
|        2|2020-01-01 00:02:00|
|        3|2020-01-01 00:03:00|
|        4|2020-01-01 00:04:00|
|        5|2020-01-01 00:05:00|
|        6|2020-01-01 00:06:00|
|        7|2020-01-01 00:07:00|
|        8|2020-01-01 00:08:00|
|        9|2020-01-01 00:09:00|
|       10|2020-01-01 00:10:00|
+---------+-------------------+
only showing top 10 rows


In [0]:
df_clean.write \
    .mode("overwrite") \
    .parquet(PATH_SILVER)

print(f"✅ Silver écrit dans : {PATH_SILVER}")

# On relit immédiatement ce qu'on vient d'écrire
# → remplace le cache() : df_clean est maintenant stable
df_clean = spark.read.parquet(PATH_SILVER)
total_silver = df_clean.count()

print(f"✅ Silver relu depuis ADLS — {total_silver} lignes confirmées")

✅ Silver écrit dans : /Volumes/workspace/default/bronze/silver/sensor_clean/
✅ Silver relu depuis ADLS — 10000 lignes confirmées


In [0]:
# ============================================================
# CELLULE 12 — VÉRIFICATION FINALE (bonne pratique pro)
# On relit le fichier qu'on vient d'écrire pour confirmer
# qu'il est bien lisible et complet — toujours faire ça en prod
# ============================================================
df_verify = spark.read.parquet(PATH_SILVER)

print('Vérification Silver (relecture) :')
print(f'  Lignes   : {df_verify.count()}')
print(f'  Colonnes : {len(df_verify.columns)}')
df_verify.show(3)

print('\n✅ Silver validé — prêt pour J6 Feature Engineering')

Vérification Silver (relecture) :
  Lignes   : 10000
  Colonnes : 15
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+------------------------+-------------+------------------+--------------+-------------------+
|sensor_id|product_id|station_type|air_temp_k|process_temp_k|rotation_speed_rpm|torque_nm|tool_wear_min|machine_failure|tool_wear_failure|heat_dissipation_failure|power_failure|overstrain_failure|random_failure|          timestamp|
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+------------------------+-------------+------------------+--------------+-------------------+
|        1|    M14860|           M|     298.1|         308.6|            1551.0|     42.8|            0|              0|                0|                       0|            0|                 0|             0|2020-01-01 00:01:00|
|  